
# Week 9 Lab — Exploring CAP Trade-offs with MongoDB

This notebook guides you through visualizing CAP theorem choices and experimenting with MongoDB read/write concerns using your Atlas cluster.



## Lab Objectives
- Revisit the CAP theorem with an interactive visualization.
- Inspect how MongoDB toggles trade-offs using read and write concerns.
- Capture latency and staleness observations from live tests.



## Prerequisites
1. Python 3.9+ environment with access to the internet.
2. Installed packages: `pymongo`, `matplotlib`, `pandas`.
3. Atlas cluster credentials (connection string, username, password).

> Tip: uncomment and run the next cell if you need to install dependencies inside the notebook runtime.


In [ ]:

# Optional setup: install required packages
# %pip install pymongo matplotlib pandas



## Imports & Configuration
Update the `ATLAS_URI` with your SRV string. Keep credentials in environment variables when possible.


In [ ]:

import os
import time

import matplotlib.pyplot as plt
import pandas as pd
from pymongo import MongoClient
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern

# Replace with environment variable lookup in production workflows
ATLAS_URI = os.getenv("ATLAS_URI", "mongodb+srv://<user>:<password>@<cluster-host>/<db>?retryWrites=true&w=majority")



## Part 1 — Visualizing CAP Priorities
The chart below positions popular datastores based on their default CAP emphasis. Adjust the weights to reflect how your workload leans.


In [ ]:

cap_weights = pd.DataFrame([
    {"System": "MongoDB Atlas", "Consistency": 0.33, "Availability": 0.33, "Partition": 0.34},
    {"System": "Apache Cassandra", "Consistency": 0.2, "Availability": 0.4, "Partition": 0.4},
    {"System": "Redis (primary)", "Consistency": 0.45, "Availability": 0.4, "Partition": 0.15},
    {"System": "Neo4j Fabric", "Consistency": 0.4, "Availability": 0.2, "Partition": 0.4},
])

fig, ax = plt.subplots(figsize=(8, 5))
for _, row in cap_weights.iterrows():
    ax.plot([row["Consistency"], row["Availability"], row["Partition"]], label=row["System"], marker="o")

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["Consistency", "Availability", "Partition tolerance"])
ax.set_ylim(0, 0.6)
ax.set_ylabel("Relative emphasis (0-0.6)")
ax.set_title("CAP Emphasis Comparison (tunable per workload)")
ax.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.show()



## Reflection Prompt
Record which system best aligns with your midterm project and note any mismatches between its defaults and your SLA requirements.


In [ ]:

cap_notes = (
    "System alignment notes:
"
    "- 
"
    "- 
"
)
print(cap_notes)



## Part 2 — Connecting to MongoDB with Tunable Concerns
This section demonstrates how to test different read and write concern combinations. Ensure the target collection exists before running writes.


In [ ]:

# Connect with majority write concern by default
client = MongoClient(ATLAS_URI, wtimeoutMS=5000)
db = client.get_default_database()
collection = db.get_collection(
    "cap_lab",
    write_concern=WriteConcern("majority"),
    read_concern=ReadConcern("local"),
)

print(f"Connected to database: {db.name}")


In [ ]:

# Seed a sample document for repeated reads
payload = {"_id": "session-001", "status": "draft", "updated": time.time()}
collection.replace_one({"_id": payload["_id"]}, payload, upsert=True)
print("Document upserted.")


In [ ]:

def read_with_preference(read_concern_level: str, delay: float = 0.0):
    start = time.perf_counter()
    coll = collection.with_options(read_concern=ReadConcern(read_concern_level))
    doc = coll.find_one({"_id": "session-001"})
    elapsed = (time.perf_counter() - start) * 1000
    if delay:
        time.sleep(delay)
    return {
        "readConcern": read_concern_level,
        "elapsed_ms": round(elapsed, 2),
        "status": doc["status"] if doc else None,
    }

scenarios = [
    read_with_preference("local"),
    read_with_preference("majority"),
]
scenarios


In [ ]:

def simulate_eventual_consistency():
    majority_coll = collection.with_options(read_concern=ReadConcern("majority"))
    majority_coll.update_one({"_id": "session-001"}, {"$set": {"status": "committed", "updated": time.time()}})

    print("Status set to 'committed' with majority write concern.")

    local_read = read_with_preference("local")
    majority_read = read_with_preference("majority")
    available_read = read_with_preference("available")

    return pd.DataFrame([local_read, majority_read, available_read])

try:
    results = simulate_eventual_consistency()
    display(results)
finally:
    collection.update_one({"_id": "session-001"}, {"$set": {"status": "draft", "updated": time.time()}})



> The `available` read concern requires replica set or sharded deployments running MongoDB 4.2+. Swap for `linearizable` or `snapshot` if your tier supports those guarantees.



## Observation Log
Summarize latency differences and any stale reads you witnessed when reducing consistency guarantees.


In [ ]:

observation_template = (
    "Latency observations:
"
    "- 
"
    "Staleness observations:
"
    "- 
"
    "Risk mitigations:
"
    "- 
"
)
print(observation_template)



## Cleanup
Run the following cell when finished to remove lab artifacts (optional).


In [ ]:

collection.delete_one({"_id": "session-001"})
client.close()
print("Lab artifacts removed and client connection closed.")



## Next Steps
- Adapt the automation pattern into the `sample_mongodb_script.js` workflow.
- Compare notebook observations with the schema design patterns course to plan Week 10 prep.
- Push your reflections to the LMS discussion ahead of the next class.
